# Reprodução em Google Colab

Este notebook executa parte dos experimentos do TCC em uma GPU do Google Colab, sem
depender da plataforma usada originalmente, e compara os resultados com os arquivos
salvos em `data/final/`. Os dois modos executados são:

- `behavior`: comportamento de indentação nos 384 pares de transferência e nas 96 lacunas de código natural;
- `bracket`: validação causal das candidatas de delimitadores nos 96 pares de confirmação.

**Antes de começar:** em *Ambiente de execução > Alterar o tipo de ambiente de execução*, selecione uma GPU (por exemplo, T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Obter o código

O repositório público do trabalho é clonado do GitHub.

In [ ]:
REPO = "goerll/mechinterp-code-lm"

import os, subprocess
if not os.path.exists("mechinterp-code-lm"):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git"], check=True)
%cd mechinterp-code-lm


## 2. Instalar as versões fixadas

As versões são as mesmas usadas nos experimentos originais. Se o Colab pedir para
reiniciar o ambiente após a instalação, reinicie e continue a partir da seção 3.
O `torchvision` pré-instalado no Colab é removido porque foi compilado para outra versão
do PyTorch e não é usado pelos experimentos.

In [ ]:
import sys
packages = [
    "torch==2.6.0",
    "transformers==4.57.6",
    "transformer-lens==2.17.0",
    "typeguard==4.5.1",
    "einops==0.8.2",
    "pandas==2.3.3",
    "tqdm==4.67.3",
]
# numpy 1.26.4 não tem pacotes para Python 3.13 ou mais recente; nesse caso o pip
# escolhe uma versão compatível (o numpy não participa dos cálculos do modelo).
if sys.version_info < (3, 13):
    packages.append("numpy==1.26.4")
print("Python", sys.version.split()[0])
!pip install -q {" ".join(packages)}
# O Colab traz torchvision/torchaudio compilados para outra versão do PyTorch; eles
# não são usados aqui e, se ficarem instalados, impedem a importação do transformers.
!pip uninstall -y -q torchvision torchaudio

## 3. Executar os experimentos

Cada execução baixa o SantaCoder na primeira vez (cerca de 5 GB) e grava o resultado
em `data/rerun/`. Os resultados da verificação original, feita em uma GPU T4, estão em `data/colab/`. Na GPU T4, o modo `behavior` leva alguns minutos e o modo `bracket`,
cerca de dez a quinze minutos.

In [ ]:
%cd /content/mechinterp-code-lm
!python -m scripts.run_experiment --mode behavior --save-path data/rerun/behavior_run.json

In [ ]:
!python -m scripts.run_experiment --mode bracket --save-path data/rerun/bracket_validation.json

## 4. Comparar com os resultados salvos

A comparação percorre todos os valores numéricos dos dois arquivos (margens e efeitos
das intervenções). Diferenças pequenas são esperadas entre GPUs diferentes, por causa da
ordem das operações em ponto flutuante; a tolerância padrão é de 0,001 *logit*.

In [ ]:
!python -m scripts.compare_runs data/final/behavior_run.json data/rerun/behavior_run.json

In [ ]:
!python -m scripts.compare_runs data/final/bracket_validation.json data/rerun/bracket_validation.json

## 5. Baixar os resultados

Gera um arquivo compactado com os resultados desta execução, para arquivamento junto
ao repositório.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader > data/rerun/gpu.txt
!zip -qr colab_results.zip data/rerun
from google.colab import files
files.download("colab_results.zip")